# KNSB Thermochemical Justification

## 1. Setting up the environment

I'll be using a powerful tool known as [PyProPEP](https://github.com/jonnydyer/pypropep/tree/master) for this task. It's a Python interface to [CProPEP](https://rocketworkbench.sourceforge.net/) (*an improvement on ForTran ProPEP*). It calculates everything accurately and efficiently. Also ensure you've set up Jupyter on your machine.

First, install the libraries:

In [ ]:
%pip install --no-cache-dir -r requirements.txt

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.9/16.9 MB 144.4 kB/s  0:02:13m0:00:0100:05
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 122.5 kB/s  0:01:28m0:00:0300:05
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 110.0 kB/s  0:01:17m0:00:0200:04
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 113.1 kB/s  0:05:41m0:00:0200:09
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 64.2 kB/s  0:01:08m0:00:0100:04m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 78.7 kB/s  0:00:106m-:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 23.6 kB/s  0:02:07m0:00:0200:08m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 61.0 kB/s  0:01:01m0:00:0200:06m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 627.5/627.5 kB 142.7 kB/s  0:00:04eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━

---
## 2. A little bit on it's Stoichiometry...
We'll now solve for the following equation:

$$
KNO_{3\left(s\right)}+C_6H_{14}O_{6\left(s\right)} \rightarrow CO_{2\left(g\right)}+CO_{\left(g\right)}+H_2O_{\left(g\right)}+H_{2\left(g\right)}+N_{2\left(g\right)}+K_2CO_{3\left(g\right)}+KOH_{\left(g\right)}
$$

> This is what annoys me with this equation. It's too spread out and at the high temperature i.e. **Adiabatic Flame Temperature** (AFT) - *that's ranges from $2100\:K$ to $3500\:K$ depending on the composite in question.* - one can intuitively tell that these extra products will collapse into the following:
> $$
\boxed{
26\,\text{KNO}_3 + 5\,\text{C}_6\text{H}_{14}\text{O}_6
\;\longrightarrow\;
17\,\text{CO}_2 + 35\,\text{H}_2\text{O} + 13\,\text{N}_2 + 13\,\text{K}_2\text{CO}_3
}
> $$

A necessary GCD reduction step after solving the null-space problem as ensures you get the minimal positive integer coefficients that represent the equation as the free variable value will be "kinda wonky".

This is implemented in `stoichiometric_justification.ipynb`.

---
## 3. Now the fun part...

I'll first start by importing the various libraries we need for this task. 


In [4]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# Fix deprecated collections imports
import collections.abc
import collections
if not hasattr(collections, 'MutableMapping'):
    collections.MutableMapping = collections.abc.MutableMapping
if not hasattr(collections, 'Mapping'):
    collections.Mapping = collections.abc.Mapping
if not hasattr(collections, 'Sequence'):
    collections.Sequence = collections.abc.Sequence

import pypropep as ppp

We'll initialize `pypropep` and set up our plotting as follows:

In [5]:
ppp.init()

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'DejaVu Sans'
})

Loaded 1921 thermo species
Loaded 1031 propellants


Then I'll define my global variables that will be used throughout the code. For this we'll need the **mechanism** which points to a yaml file that contains the details of the Grains in question and to define some lists like the **flame temp.** and **specific impulse** that will contain the calculated values from our script

We'll then parse the data in the CSV file and get the masses of $KNO_3$ and $C_6H_{14}O_6$ and store them as a dataframe.

In [6]:
df = pd.read_csv("./share/csv/Thermochemical_Justification.csv")

print(df.dtypes)
print(df[['Mass KNO3 (g)','Mass Sorbitol (g)']].head())
print(df.isna().sum())

df['Mass KNO3 (g)']    = pd.to_numeric(df['Mass KNO3 (g)'], errors='coerce')
df['Mass Sorbitol (g)'] = pd.to_numeric(df['Mass Sorbitol (g)'], errors='coerce')
df = df.dropna(subset=['Mass KNO3 (g)','Mass Sorbitol (g)'])

df.head()

KNO3 (%)             float64
Sorbitol (%)         float64
Mass KNO3 (g)        float64
Mass Sorbitol (g)    float64
C*                   float64
Density              float64
Chamber Cp/Cv        float64
Chamber Temp         float64
Isp*                 float64
Molecular Weight     float64
dtype: object
   Mass KNO3 (g)  Mass Sorbitol (g)
0         726.80             986.83
1         730.43             981.90
2         734.07             976.96
3         737.70             972.03
4         741.34             967.09
KNO3 (%)               0
Sorbitol (%)           0
Mass KNO3 (g)          0
Mass Sorbitol (g)      0
C*                   161
Density              161
Chamber Cp/Cv        161
Chamber Temp         161
Isp*                 161
Molecular Weight     161
dtype: int64


,KNO3 (%),Sorbitol (%),Mass KNO3 (g),Mass Sorbitol (g),C*,Density,Chamber Cp/Cv,Chamber Temp,Isp*,Molecular Weight
0,50.00,50.00,726.80,986.83,NaN,NaN,NaN,NaN,NaN,NaN
1,50.25,49.75,730.43,981.90,NaN,NaN,NaN,NaN,NaN,NaN
2,50.50,49.50,734.07,976.96,NaN,NaN,NaN,NaN,NaN,NaN
3,50.75,49.25,737.70,972.03,NaN,NaN,NaN,NaN,NaN,NaN
4,51.00,49.00,741.34,967.09,NaN,NaN,NaN,NaN,NaN,NaN


Now that's sorted, we can calculate the following data as per the csv file:

#### 1. **Density** ($\rho$)

We'll be using the following formula:

$$
\rho_p = \frac{1}{\frac{f_0}{\rho_0}+\frac{f_1}{\rho_1}}
$$

where $f_n$ is the **mass fraction** of the propellant component and $\rho_n$ is the respective density.

#### 2. **Chamber $c_p/c_v$** ($k$ or $\gamma$)

This is the **specific heat ratio** and it is used to determine the chamber pressure. It's given by:

$$
k = \frac{1}{1-\frac{R}{\frac{X}{1-X}C_s+C_p}}
$$

where $R$ is the **universal gas constant**, $\frac{X}{1-X}$ is the **mole fraction of condensed phase products**, $C_s$ is the **specific heat of the mixture of condensed phase products** and $C_p$ is the **specific heat of the mixture of gaseous products**.

It should be noted that this is for a **2-phase flow** where we have both condensed(*solid*) and gaseous phase products from this combustion.(Ikiara, 2025)

#### 3. **Chamber velocity** ($C^*$)

This is essential in determining the speed at which the propellant will burn at. It's given by:

$$
C^*=\sqrt{\frac{R\:T_o}{k\left(\frac{2}{k+1}\right)^{\frac{k+1}{k-1}}}}
$$

where $T_o$ is the **chamber temperature**.

#### 4. **Specific impulse** ($I_{sp}$)

This is the crucial factor that determines the thrust of our SRM. It is given by:

$$
I_{sp}=\frac{1}{g}\sqrt{2T_o\left(\frac{R}{M}\right)(\frac{k}{k-1})\left[1-\frac{P_e}{P_o}\right]^{\frac{k-1}{k}}}
$$

where $g$ is the **accelearation due to gravity**, $M$ is the **molecular mass of the KNSB grain**, $P_e$ and $P_o$ are the **exit** and **chamber pressures**.

In [ ]:
# Components
sorb = ppp.PROPELLANTS['SORBITOL']
kno3 = ppp.PROPELLANTS['POTASSIUM NITRATE']

# Component densities (g/cm³)
RHO_KNO3     = 2.109
RHO_SORBITOL = 1.489

# Motor operating conditions
P_CHAMBER_ATM = 68.0   # ~1000 psi, typical KNSB
P_EXIT_ATM    = 1.0    # sea level

# Accounting for the two-phase flow as (Nakka, 2025)


# ── Simulation loop ──────────────────────────────────────────────────────────
isps, chamber_temps, c_stars, isexs = [], [], [], []
densities, cps, cvs, gammas, of_ratios = [], [], [], [], []

From this data we can show the relationship between $I_{sp}$ and mass of $KNO_3$ used:

In [ ]:
# SORB_PURITY   = 0.7365
# MIN_MASS_G    = 10.0
# G0            = 9.80665

# def _fill_nan():
#     for lst in [isps, c_stars, chamber_temps, gammas, cps, cvs, of_ratios, densities]:
#         lst.append(np.nan)

# for i, row in df.iterrows():
    
#     m_fu          = m_fu_liquid * SORB_PURITY

#     if m_ox < MIN_MASS_G or m_fu < MIN_MASS_G:
#         print(f"[SKIP] KNO3={kno3_pct:.2f}% — mass too low for solver (m_ox={m_ox:.1f}g, m_fu={m_fu:.1f}g)")
#         _fill_nan()    # see helper below
#         continue

    
#     of_ratio = m_ox / m_fu if m_fu != 0 else 0
#     of_ratios.append(round(of_ratio, 4))


for _, row in df.iterrows():
    pct_ox = row['KNO3 (%)']
    pct_fu = row['Sorbitol (%)']
    m_ox    = row['Mass KNO3 (g)']
    m_fu    = row['Mass Sorbitol (g)']

    print(f"[INFO] Siulating KNO3={kno3_pct}%")
    
    try:
        p = ppp.FrozenPerformance()
        p.add_propellants_by_mass([(sorb, m_fu), (kno3, m_ox)])
        p.set_state(P=P_CHAMBER_ATM, Pe=P_EXIT_ATM)
                    
        # isps.append(round(p.performance.Isp, 4))
        chamber_temps.append(round(p.properties[0].T, 4))
#         c_stars.append(round(e.performance.cstar, 4))
#         cps.append(round(e.properties[0].Cp, 4))
#         cvs.append(round(e.properties[0].Cv, 4))
#         isexs.append(round(e.properties[0].Isex, 4))
#         gammas.append(round(e.properties[0].Cp / e.properties[0].Cv, 4))
    except Exception as e:
        print(f"[FAIL] KNO3={pct_ox:.2f}% Sorbitol={pct_fu:.2f}% — solver error: {e}")
        continue
    
    f_ox  = pct_ox   / 100.0
    f_fu  = pct_fu / 100.0
    rho_p = 1.0 / (f_ox / RHO_KNO3 + f_fu / RHO_SORBITOL)
    densities.append(rho_p)

print(f"[INFO] Simulation completed for {len(isps)} cases.")

LU: matrix is singular, no unique solution.
LU: matrix is singular, no unique solution.
LU: matrix is singular, no unique solution.
LU: matrix is singular, no unique solution.
LU: matrix is singular, no unique solution.
LU: matrix is singular, no unique solution.
LU: matrix is singular, no unique solution.
LU: matrix is singular, no unique solution.


: 

We'll then write the results back to the DataFrame and display it for confirmation:

In [ ]:
n = min(len(df), len(c_stars), len(densities), len(cps), len(cvs), len(chamber_temps), len(isps), len(of_ratios))
df = df.iloc[:n].copy()

df['C*']            = c_stars[:n]
df['Density']       = densities[:n]
df['Chamber Cp']    = cps[:n]
df['Chamber Cv']    = cvs[:n]
df['Chamber Temp']  = chamber_temps[:n]
df['Isp*']          = isps[:n]
df['Chamber Cp/Cv'] = gammas[:n]
df['Isex']          = isexs[:n]
df['O/F']           = of_ratios[:n]

df.to_csv("./share/csv/Thermochemical_Justification.csv", index=False)
df.head()

We can also outline the relationship between the flame temperatures and mass of $KNO_3$ used:

In [ ]:
import os

kno3_pct = df['KNO3 (%)']
sweet_spot_idx = df[df['KNO3 (%)'] == 65.0].index[0]

fig = plt.figure(figsize=(15, 4))
gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.35)

panels = [
    (gs[0], kno3_pct, isps,          'Specific Impulse (s)',      'Isp vs KNO₃ Loading'),
    (gs[1], kno3_pct, chamber_temps, 'Chamber Temperature (K)',   'T_chamber vs KNO₃ Loading'),
    (gs[2], kno3_pct, c_stars,       'Characteristic Velocity C* (m/s)', 'C* vs KNO₃ Loading'),
]

for spec, x, y, ylabel, title in panels:
    ax = fig.add_subplot(spec)
    ax.plot(x, y, 'o-', lw=1.5, ms=4, color='steelblue')
    ax.axvline(x=kno3_pct[sweet_spot_idx], color='tomato',
               linestyle='--', lw=1.2, label=f"Optimal: {kno3_pct[sweet_spot_idx]:.2f}%")
    ax.set_xlabel('KNO₃ Loading (%)')
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=8)

plt.suptitle('KNSB Propellant Performance Sweep', fontweight='bold')

# Create directories if they don't exist
output_dir = './share/images'
os.makedirs(output_dir, exist_ok=True)

# Save figure
plt.savefig(os.path.join(output_dir, 'knsb_performance_sweep.png'), bbox_inches='tight')
plt.show()

In [ ]:
import os

opt_target = 1.857
opt_idx = df['O/F'].sub(opt_target).abs().idxmin()
opt_of = df.loc[opt_idx, 'O/F']

fig = plt.figure(figsize=(15, 4))
gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.35)

panels = [
    (gs[0], isps,          'Specific Impulse (s)',          'Isp'),
    (gs[1], chamber_temps, 'Chamber Temperature (K)',        'T_c'),
    (gs[2], c_stars,       'Characteristic Velocity (m/s)', 'C*'),
]

for spec, y, ylabel, label in panels:
    ax = fig.add_subplot(spec)
    ax.plot(of_ratios, y, 'o-', lw=1.5, ms=4, color='steelblue', label=label)
    ax.axvline(opt_of, color='tomato', ls='--', lw=1.2,
               label=f'Optimum O/F = {opt_of:.3f}')
    ax.set_xlabel('O/F Ratio (KNO₃ / Sorbitol)')
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=8)

plt.suptitle('KNSB Performance vs. O/F Ratio', fontweight='bold')
plt.tight_layout()

# Create directories if they don't exist
output_dir = './share/images'
os.makedirs(output_dir, exist_ok=True)

# Save figure
plt.savefig(os.path.join(output_dir, 'knsb_performance_of_ratio.png'), bbox_inches='tight')
plt.show()

---
## 4. What we can learn from this data

We'll find that with more mass of $KNO_3$ used, do these critical factors i.e. $I_{sp}$, chamber temperature and $C^*$ increase. In reality, KNSB grains made with such concentrations of $KNO_3$ are harder to process and for the motor casing to handle.

We'll also find that with less mass of $KNO_3$ used, the motor is cooler but less energetic. We can see that at $65\%:35\%$ i.e. when mass of $KNO_3$ used is $944.84g$, we have the "[sweet spot](https://tenor.com/blpXi.gif)".

It's also worth mentioning that when too much oxidizer is used, the KNSB grain becomes brittle and sensitive to moisture while when too much fuel is used, incomplete combustion and lower $I_{sp}$ is noted. ([Nakka, 2025](https://www.nakka-rocketry.net/sorb.html))

Since it's a 2-phase flow analysis:
### 1. The gaseous phase
The dominant species from the combustion tell us the following in the table below:
| Species | Mole fraction | What it means |
| :--: | :--: | :--: |
| $H_2O$ | 0.304 | Primary hydrogen combustion product |
| $CO$ | 0.185 | Incomplete carbon oxidation |
| $H_2$ | 0.176 | Unburned hydrogen |
| $CO_2$ | 0.122 | Complete carbon oxidation |
| $N_2$ | 0.106 | Nitrogen release from $KNO_3$ |

In [ ]:
# Quick sanity check
# After the loop, run this for the optimal ratio point
p_opt = ppp.FrozenPerformance()
p_opt.add_propellants_by_mass([
    (kno3,     df.loc[opt_idx, 'Mass KNO3 (g)']),
    (sorb, df.loc[opt_idx, 'Mass Sorbitol (g)']),
])
p_opt.set_state(P=68., Pe=1.0)

print("=== Optimal KNSB (O/F = {:.3f}) ===".format(opt_of))
print("\nGaseous chamber products:")
import pprint
pprint.pprint(p_opt.composition['chamber'][:8])
print("\nCondensed chamber products (K₂CO₃ should appear here):")
pprint.pprint(p_opt.composition_condensed['chamber'])

---
## 5. What's in-store for the future of this simulation
We plan on performing a CFD Simulation of 1-D isentropic nozzle flow i.e.:

$$
M_e = f(P_c, P_e, \gamma), v_e = \sqrt{\frac{2\gamma}{\gamma-1}\frac{RT_e}{M_w}\left[1-\left(\frac{P_e}{P_c}\right)^{\frac{\gamma-1}{\gamma}}\right]}
$$

using OpenFOAM and ANSYS programs hence the 2 folders in the `/share` directory. This is pretty taxing so far and will require some time.


---
## 6. In conclusion...

We justified the optimal KNSB ratio i.e. $65\%:35\%$, as it provides a high specific impulse with efficient combustion, *indicated by $I_{sp}$ and $C^*$*, while maintaining safe thermal characteristics and physical integrity. It is a well-characterized formulation that makes it a reliable and practical choice for our SRM.

## References

[1] R. Nakka, "KNSB Propellant," Richard Nakka's Experimental Rocketry Web Site, 2025. [Online]. Available: https://www.nakka-rocketry.net/sorb.html

[2] R. Nakka, "Solid Rocket Motor Theory - Two-phase flow," Richard Nakka's Experimental Rocketry Web Site, 2025. [Online]. Available: https://www.nakka-rocketry.net/th_2phf.htm

[3] R. Nakka, "Solid Rocket Motor Theory -- Impulse and C-star," Richard Nakka's Experimental Rocketry Web Site, 2025. [Online]. Available: https://www.nakka-rocketry.net/th_imp.html

[4] R. Nakka, "Two-phase flow theory," Richard Nakka's Experimental Rocketry Web Site, Oct. 28, 2011. [Online]. Available: nakka-rocketry.net. [Accessed: May 31, 2026].

[5] R. Nakka, "Potassium nitrate/sorbitol (KNSB) propellant performance characteristics," Richard Nakka's Experimental Rocketry Web Site, Mar. 22, 2011. [Online]. Available: nakka-rocketry.net. [Accessed: May 31, 2026].

[5] J. Bonnie, J. Zehe, and S. Gordon, "NASA Glenn Coefficients for Calculating Thermodynamic Properties of Individual Species," NASA/TP—2002-211556, Glenn Research Center, Cleveland, 2002.

[6] T. McReary, Experimental Composite Propellant: An Introduction To Properties And Preparation OF Composite Propellants: Design, Construction, Testing, and Characteristics Of Small Rocket Motors, 1st ed. 2020.

[7] D. Mazza and E. Canuto, Fundamental Chemistry with MATLAB. Oxford Publishing, 2022.